In [8]:
import numpy as np
import math
import matplotlib.pyplot as plt
from ipynb.fs.full.Algoritmo_DFT_e_IDFT import dft_imp, idft_imp
from ipynb.fs.defs.Algoritmo_FFT import fft_radix2
# ==============================================================================
# 1. OVERLAP-AND-ADD (OLA)
# ==============================================================================
def overlap_add(x, h, L, dft=dft_imp, idft=idft_imp, verbose=False):
    """
    Convolução linear de x (longo) com h (curto) pelo método overlap-and-add.
 
    x : sinal de entrada (longo)
    h : resposta ao impulso do filtro (curta), M = len(h)
    L : tamanho de cada bloco de x
    Tamanho da DFT de cada bloco: N = L + M - 1 (sem aliasing).
 
    Ideia:
      1. Divide x em blocos de L amostras, sem sobreposição.
      2. Convolui cada bloco com h (via DFT de N pontos). Cada saída tem N amostras.
      3. Posiciona cada saída no início do seu bloco e SOMA as caudas que se sobrepõem.
    """
    x = np.asarray(x, dtype=float)
    h = np.asarray(h, dtype=float)
    M = len(h)
    N = L + M - 1                      # tamanho da DFT
    Ly = len(x) + M - 1                # tamanho da saída linear total
 
    H = dft(np.pad(h, (0, N - M)))     # DFT do filtro calculada uma única vez
    y = np.zeros(Ly + N)               # folga no final para somar o último bloco
 
    for i, inicio in enumerate(range(0, len(x), L)):
        bloco = x[inicio:inicio + L]                       # último bloco pode ser menor
        X = dft(np.pad(bloco, (0, N - len(bloco))))
        y_bloco = idft(X * H).real                         # N amostras
 
        y[inicio:inicio + N] += y_bloco                    # soma a parte sobreposta
 
        if verbose:
            print(f"Bloco {i}: x[{inicio}:{inicio + len(bloco)}] = {bloco}")
            print(f"   saida do bloco ({N} pts) = {np.round(y_bloco, 4)}")
            print(f"   somada em y[{inicio}:{inicio + N}]")
 
    return y[:Ly]
 
 
# ==============================================================================
# 2. OVERLAP-AND-SAVE (OLS)
# ==============================================================================
def overlap_save(x, h, N, dft=dft_imp, idft=idft_imp, verbose=False):
    """
    Convolução linear de x (longo) com h (curto) pelo método overlap-and-save.
 
    x : sinal de entrada (longo)
    h : resposta ao impulso do filtro (curta), M = len(h)
    N : tamanho da DFT e de cada bloco de entrada (N >= M)
    Amostras novas por bloco: L = N - M + 1.
 
    Ideia:
      1. Coloca M-1 zeros no início de x.
      2. Pega blocos de N amostras, andando L amostras de cada vez (os blocos
         se SOBREPÕEM em M-1 amostras).
      3. Faz a convolução CIRCULAR de N pontos de cada bloco com h.
      4. As primeiras M-1 amostras da saída estão contaminadas por aliasing e são
         DESCARTADAS. As outras L são as amostras corretas e são SALVAS.
    """
    x = np.asarray(x, dtype=float)
    h = np.asarray(h, dtype=float)
    M = len(h)
    if N < M:
        raise ValueError("N precisa ser >= len(h)")
    L = N - M + 1                      # amostras úteis por bloco
    Ly = len(x) + M - 1                # tamanho da saída linear total
    n_blocos = int(np.ceil(Ly / L))
 
    # M-1 zeros na frente e zeros no fim para completar o último bloco
    x_pad = np.concatenate([np.zeros(M - 1), x])
    x_pad = np.pad(x_pad, (0, n_blocos * L + M - 1 - len(x_pad)))
 
    H = dft(np.pad(h, (0, N - M)))     # DFT do filtro calculada uma única vez
    y = np.zeros(n_blocos * L)
 
    for i in range(n_blocos):
        inicio = i * L
        bloco = x_pad[inicio:inicio + N]                   # N amostras (com sobreposição)
        X = dft(bloco)
        y_bloco = idft(X * H).real                         # convolução circular de N pontos
 
        y[inicio:inicio + L] = y_bloco[M - 1:]             # descarta M-1, salva L
 
        if verbose:
            print(f"Bloco {i}: x_pad[{inicio}:{inicio + N}] = {bloco}")
            print(f"   circular ({N} pts) = {np.round(y_bloco, 4)}")
            print(f"   descarta {M - 1} primeira(s) e salva {np.round(y_bloco[M - 1:], 4)}")
 
    return y[:Ly]


In [9]:
# ==============================================================================
# 3. TESTES
# ==============================================================================
if __name__ == "__main__":
    # --- Teste 1: sinal pequeno, passo a passo ---
    x = np.array([3, 1, 4, 1, 5, 9, 2, 6], dtype=float)
    h = np.array([1, 2, 1], dtype=float)
    ref = np.convolve(x, h)
 
    print("=== TESTE 1: x de 8 amostras, h = [1, 2, 1] ===")
    print("\n--- OVERLAP-AND-ADD (L = 3, N = 5) ---")
    y_ola = overlap_add(x, h, L=3, verbose=True)
    print("\n--- OVERLAP-AND-SAVE (N = 5, L = 3) ---")
    y_ols = overlap_save(x, h, N=5, verbose=True)
 
    print("\nReferência np.convolve:", ref)
    print("Overlap-add           :", np.round(y_ola, 4))
    print("Overlap-save          :", np.round(y_ols, 4))
    print(f"Erro máx OLA: {np.max(np.abs(y_ola - ref)):.2e}")
    print(f"Erro máx OLS: {np.max(np.abs(y_ols - ref)):.2e}")
 
    # --- Teste 2: sinal mais longo e vários tamanhos de bloco ---
    rng = np.random.default_rng(0)
    x2 = rng.integers(-5, 10, size=100).astype(float)
    h2 = np.array([1, 2, 3, 2, 1], dtype=float)
    ref2 = np.convolve(x2, h2)
 
    print("\n=== TESTE 2: x de 100 amostras, h de 5 coeficientes ===")
    print(f"{'L (OLA)':>8} {'N (OLS)':>8} {'erro OLA':>12} {'erro OLS':>12}")
    for L in (1, 7, 10, 16, 50, 100, 150):
        N = L + len(h2) - 1
        e1 = np.max(np.abs(overlap_add(x2, h2, L) - ref2))
        e2 = np.max(np.abs(overlap_save(x2, h2, N) - ref2))
        print(f"{L:>8} {N:>8} {e1:>12.2e} {e2:>12.2e}")
 


=== TESTE 1: x de 8 amostras, h = [1, 2, 1] ===

--- OVERLAP-AND-ADD (L = 3, N = 5) ---
Bloco 0: x[0:3] = [3. 1. 4.]
   saida do bloco (5 pts) = [3. 7. 9. 9. 4.]
   somada em y[0:5]
Bloco 1: x[3:6] = [1. 5. 9.]
   saida do bloco (5 pts) = [ 1.  7. 20. 23.  9.]
   somada em y[3:8]
Bloco 2: x[6:8] = [2. 6.]
   saida do bloco (5 pts) = [ 2. 10. 14.  6.  0.]
   somada em y[6:11]

--- OVERLAP-AND-SAVE (N = 5, L = 3) ---
Bloco 0: x_pad[0:5] = [0. 0. 3. 1. 4.]
   circular (5 pts) = [9. 4. 3. 7. 9.]
   descarta 2 primeira(s) e salva [3. 7. 9.]
Bloco 1: x_pad[3:8] = [1. 4. 1. 5. 9.]
   circular (5 pts) = [24. 15. 10. 11. 20.]
   descarta 2 primeira(s) e salva [10. 11. 20.]
Bloco 2: x_pad[6:11] = [5. 9. 2. 6. 0.]
   circular (5 pts) = [11. 19. 25. 19. 14.]
   descarta 2 primeira(s) e salva [25. 19. 14.]
Bloco 3: x_pad[9:14] = [6. 0. 0. 0. 0.]
   circular (5 pts) = [ 6. 12.  6.  0.  0.]
   descarta 2 primeira(s) e salva [6. 0. 0.]

Referência np.convolve: [ 3.  7.  9. 10. 11. 20. 25. 19. 14.  6.]